In [ ]:
#| hide

%load_ext autoreload
%autoreload 2

# Google's MCP Toolbox

> In this notebook, we will explore how to configure and use various tools available in Google's MCP Toolbox. These are defined in `tools.yaml` configuration file.

In [ ]:
#| default_exp toolbox

In [ ]:
#| export

from toolbox_core import ToolboxSyncClient
from agents import function_tool
from thucy.configuration import config

In [ ]:
#| export

class GenAIToolboxMCP:
    """Manager for a shared ToolboxSyncClient connection."""
    
    def __init__(self):
        self.client = None
    
    def connect(self):
        self.client = ToolboxSyncClient(config.genai_server_url)
    
    def close(self):
        if self.client:
            self.client.close()
    
    def _load_raw_toolset(self, toolset_name: str):
        """Load raw toolset from client."""
        if not self.client:
            raise RuntimeError("Call connect() first")
        return self.client.load_toolset(toolset_name)
    
    def load_toolset(self, toolset_name: str):
        """Load toolset and wrap tools for OpenAI SDK."""
        return [function_tool(tool) for tool in self._load_raw_toolset(toolset_name)]

We first create a singleton instance of `GenAIToolboxMCP`.

In [ ]:
#| exports
#| notest
genai_mcp = GenAIToolboxMCP()

Then, we preconfigure the tools to redirect to the **toolsets** with the prefix of **seattle**. For example, after the following command, the SQL expert agent will subscribe to the tooset **seattle-sql**.

In [ ]:
#| notest
genai_mcp.connect()

Let's load some tools and see what is going on. First, we will load **sql** tools:

In [ ]:
#| notest
sql_tools = genai_mcp._load_raw_toolset("seattle-sql")
sql_tools

[<toolbox_core.sync_tool.ToolboxSyncTool>]

The length of the loaded tools should be 1, as we have 1 SQL toolsets in the `seattle` toolset: `postgres_seattle_execute_sql`.

In [ ]:
#| notest
len(sql_tools)

1

Let's run some SQL using a tool. **They are directly callable**!

In [ ]:
#| notest
sql_tools[0]("select * from crime_data limit 1;")

'[{"beat":"B1","block_address":"-","latitude":"-1.0","longitude":"-1.0","neighborhood":"-","nibrs_crime_against_category":"ANY","nibrs_group_ab":"B","nibrs_offense_code":"90Z","nibrs_offense_code_description":"All Other Offenses","offense_category":"ALL OTHER","offense_date":"2015-01-21T08:00:00Z","offense_id":7681626833,"offense_sub_category":"ALL OTHER","precinct":"North","report_datetime":"2015-01-27T13:43:00Z","report_number":"2015-030183","reporting_area":"5065","sector":"B","shooting_type_group":"-"}]'

Let's load the **schema** toolset and play around with it.

In [ ]:
#| notest
schema_tools = genai_mcp._load_raw_toolset("seattle-schema")

It is going to show 1 tools again. 

In [ ]:
#| notest
len(schema_tools)

1

In [ ]:
#| notest
schema_tools[0]('crime_data', 'simple')

'[{"object_details":{"name":"crime_data"},"object_name":"crime_data","schema_name":"public"}]'

In [ ]:
#| notest
schema_tools[0]('crime_data', 'detailed')

'[{"object_details":{"columns":[{"column_comment":null,"column_default":null,"column_name":"report_number","data_type":"text","is_not_nullable":false,"ordinal_position":1},{"column_comment":null,"column_default":null,"column_name":"report_datetime","data_type":"timestamp without time zone","is_not_nullable":false,"ordinal_position":2},{"column_comment":null,"column_default":null,"column_name":"offense_id","data_type":"bigint","is_not_nullable":false,"ordinal_position":3},{"column_comment":null,"column_default":null,"column_name":"offense_date","data_type":"timestamp without time zone","is_not_nullable":false,"ordinal_position":4},{"column_comment":null,"column_default":null,"column_name":"nibrs_group_ab","data_type":"text","is_not_nullable":false,"ordinal_position":5},{"column_comment":null,"column_default":null,"column_name":"nibrs_crime_against_category","data_type":"text","is_not_nullable":false,"ordinal_position":6},{"column_comment":null,"column_default":null,"column_name":"offens

We can also load the toolset used by OpenAI Agents (wrapped in tool funcs)

In [ ]:
#| notest
genai_mcp.close()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()